In [5]:
# ============================================================
# FinIntent: ViSK for Hindi/Hinglish Financial Podcast Data
# Google Colab Implementation
# Based on: "Visual Semantic Knowledge Discovery for
#            Multimodal Intent Recognition" (IEEE Access 2025)
# ============================================================
# HOW TO USE:
#   1. Upload this file to Colab or copy cell by cell
#   2. Make sure Google Drive is mounted
#   3. Update DATASET_PATH and VIDEO_FOLDER in Cell 2
#   4. Run cells top to bottom
# ============================================================


# ==============================================================
# CELL 1: Install Dependencies
# ==============================================================
# @title Install Required Packages
# Run this cell first. Colab will restart — that is normal.

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("torchvision>=0.15.0")
install("transformers==4.38.0")
install("openpyxl")          # for reading Excel files
install("opencv-python-headless")
install("scikit-learn")
install("tqdm")
install("easydict")

print("✅ All packages installed.")


# ==============================================================
# CELL 2: Mount Google Drive & Set Paths
# ==============================================================
# @title Mount Drive & Configure Paths

from google.colab import drive
drive.mount('/content/drive')

import os

# ---------------------------------------------------------------
# 👉 UPDATE THESE PATHS to match your Google Drive structure
# ---------------------------------------------------------------
# Example: if your files are in "My Drive/Capstone/Dataset.xlsx"
#          set DATASET_PATH = "/content/drive/MyDrive/Capstone/Dataset.xlsx"

DATASET_PATH  = "/content/drive/MyDrive/Capstone2"           # folder containing Dataset xlsx
EXCEL_FILE    = os.path.join(DATASET_PATH, "Dataset.xlsx") # your Excel file
VIDEO_FOLDER  = "/content/drive/MyDrive/Capstone2/Video"             # folder containing .mp4 clips
OUTPUT_DIR    = "/content/drive/MyDrive/Capstone2"  # where to save model/results

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Paths configured.\n   Excel : {EXCEL_FILE}\n   Videos: {VIDEO_FOLDER}")


# ==============================================================
# CELL 3: Imports & Global Config
# ==============================================================
# @title Imports

import os, math, logging, random, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn import DataParallel
from functools import partial
from easydict import EasyDict
from tqdm import tqdm, trange
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score)
from transformers import (AutoTokenizer, AutoModel,
                          AdamW, get_linear_schedule_with_warmup)
from torchvision.models.swin_transformer import (PatchMerging,
                                                  SwinTransformerBlock)
from torchvision.models.video.swin_transformer import (ShiftedWindowAttention3d,
                                                        PatchEmbed3d)
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {DEVICE}")


# ==============================================================
# CELL 4: Hyperparameters / Args
# ==============================================================
# @title Hyperparameters

args = EasyDict({
    # ── Paths ────────────────────────────────────────────────
    "excel_file"          : EXCEL_FILE,
    "video_folder"        : VIDEO_FOLDER,
    "output_dir"          : OUTPUT_DIR,
    "pretrained_video_path": None,   # set to local path if you download Swin weights

    # ── Data ─────────────────────────────────────────────────
    "text_col"            : "Hinglish Text",   # or "Hindi Text"
    "label_col"           : "Label",
    "video_id_col"        : "Video ID",
    "start_col"           : "Start time",
    "end_col"             : "End time",
    "max_seq_len"         : 64,        # BERT token length
    "num_frames"          : 16,        # frames sampled per clip
    "frame_size"          : 224,       # H × W fed to Swin

    # ── Model ─────────────────────────────────────────────────
    # Video Swin Transformer config (tiny variant — fits Colab GPU)
    "patch_size"          : [2, 4, 4],
    "embed_dim"           : 96,
    "depths"              : [2, 2, 6, 2],
    "num_heads"           : [3, 6, 12, 24],
    "window_size"         : [2, 7, 7],
    "mlp_ratio"           : 4.0,
    "dropout"             : 0.1,
    "attention_dropout"   : 0.1,
    "stochastic_depth_prob": 0.1,
    "sigma_init_decay"    : 0.05,      # α in paper (scaling factor)
    "iterations"          : 2,         # J — perturb & avg this many times
    "channels_1"          : 128,       # PerturbationNet hidden channels
    "channels_2"          : 64,
    "perturb_dropout"     : 0.1,

    # ── Training ──────────────────────────────────────────────
    "train_batch_size"    : 2,         # keep small for Colab
    "eval_batch_size"     : 2,
    "num_train_epochs"    : 30,
    "lr"                  : 2e-5,
    "weight_decay"        : 0.01,
    "warmup_proportion"   : 0.1,
    "grad_clip"           : 1.0,
    "wait_patience"       : 5,         # early stopping patience
    "eval_monitor"        : "weighted_f1",
    "seed"                : 42,

    # ── Text Model ────────────────────────────────────────────
    # MuRIL handles Hindi + Hinglish (code-mixed)
    "text_model_name"     : "google/muril-base-cased",

    # ── Filled automatically below ────────────────────────────
    "num_labels"          : None,
    "num_train_examples"  : None,
    "feat_size"           : None,
})

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)
print("✅ Args set.")


# ==============================================================
# CELL 5: Load & Explore Dataset
# ==============================================================
# @title Load Excel Dataset

df = pd.read_excel(args.excel_file)
print(f"Loaded {len(df)} rows. Columns: {list(df.columns)}")
print(df.head(3))

# ── Label encoding ────────────────────────────────────────────
unique_labels = sorted(df[args.label_col].dropna().unique().tolist())
label2id      = {l: i for i, l in enumerate(unique_labels)}
id2label      = {i: l for l, i in label2id.items()}
df["label_id"] = df[args.label_col].map(label2id)

args.num_labels = len(unique_labels)
print(f"\n✅ {args.num_labels} intent classes:")
for i, l in id2label.items():
    print(f"   [{i:2d}] {l}")


# ==============================================================
# CELL 6: Split Dataset (70 / 15 / 15) - Non-stratified
# ==============================================================
from sklearn.model_selection import train_test_split

df_clean = df.dropna(subset=[args.label_col, args.text_col]).reset_index(drop=True)

# We removed stratify=... to avoid the ValueError with single-member classes
train_df, temp_df = train_test_split(df_clean, test_size=0.30,
                                     random_state=args.seed)
val_df,  test_df  = train_test_split(temp_df,  test_size=0.50,
                                     random_state=args.seed)

args.num_train_examples = len(train_df)
print(f"✅ Split (No Stratification) — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


# ==============================================================
# CELL 7: Video Utilities
# ==============================================================
# @title Video Frame Extractor

def extract_frames(video_path: str,
                   start_sec: float,
                   end_sec: float,
                   num_frames: int = 16,
                   frame_size: int = 224) -> torch.Tensor:
    """
    Opens a video file, seeks to [start_sec, end_sec],
    uniformly samples `num_frames` frames, resizes to
    (frame_size × frame_size), and returns a tensor of
    shape (num_frames, 3, frame_size, frame_size) in [0,1].
    Returns zeros if the file is missing or unreadable.
    """
    zero = torch.zeros(num_frames, 3, frame_size, frame_size)
    if not os.path.exists(video_path):
        return zero

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        cap.release(); return zero

    start_f = int(start_sec * fps)
    end_f   = int(end_sec   * fps)
    total_f = max(end_f - start_f, 1)

    indices = np.linspace(start_f, end_f - 1, num_frames, dtype=int)
    frames  = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            frames.append(np.zeros((frame_size, frame_size, 3), dtype=np.uint8))
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (frame_size, frame_size))
        frames.append(frame)

    cap.release()

    # Stack → (T, H, W, C) → (T, C, H, W), normalise to [0,1]
    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(0, 3, 1, 2)  # T,C,H,W
    return tensor


def find_video_file(video_folder: str, video_id: str) -> str:
    """
    Looks for a file whose name starts with video_id
    and has a common video extension. Returns path or "".
    """
    for ext in [".mp4", ".avi", ".mkv", ".mov", ".MP4"]:
        path = os.path.join(video_folder, video_id + ext)
        if os.path.exists(path):
            return path
    # Also try exact match without extension assumption
    for fname in os.listdir(video_folder):
        if fname.startswith(video_id):
            return os.path.join(video_folder, fname)
    return ""

print("✅ Video utilities ready.")


## ==============================================================
# CELL 8: FinIntent Dataset (Fixed for Time Objects)
# ==============================================================
import datetime

class FinIntentDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, args: EasyDict, split: str = "train"):
        self.df        = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.args      = args
        self.split     = split

    def _to_seconds(self, t):
        """Helper to convert datetime.time, str, or float to total seconds."""
        if pd.isna(t):
            return 0.0
        if isinstance(t, (int, float)):
            return float(t)
        if isinstance(t, datetime.time):
            return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
        if isinstance(t, str):
            try:
                # Handles "HH:MM:SS" or "MM:SS"
                parts = [float(x) for x in t.split(':')]
                if len(parts) == 3:
                    return parts[0] * 3600 + parts[1] * 60 + parts[2]
                elif len(parts) == 2:
                    return parts[0] * 60 + parts[1]
                return parts[0]
            except:
                return 0.0
        return 0.0

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Text ─────────────────────────────────────────────
        text = str(row[self.args.text_col]) if pd.notna(row[self.args.text_col]) else ""
        enc  = self.tokenizer(
            text,
            max_length=self.args.max_seq_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attn_mask = enc["attention_mask"].squeeze(0)

        # ── Video ────────────────────────────────────────────
        video_id  = str(row[self.args.video_id_col])

        # USE THE HELPER HERE 👇
        start_sec = self._to_seconds(row[self.args.start_col])
        end_sec   = self._to_seconds(row[self.args.end_col])

        # If end time is missing or same as start, default to a 3s window
        if end_sec <= start_sec:
            end_sec = start_sec + 3.0

        video_path = find_video_file(self.args.video_folder, video_id)
        video_feats = extract_frames(
            video_path, start_sec, end_sec,
            num_frames=self.args.num_frames,
            frame_size=self.args.frame_size,
        )

        # ── Label ────────────────────────────────────────────
        label_id = int(row["label_id"])

        return {
            "video_feats" : video_feats,
            "input_ids"   : input_ids,
            "attn_mask"   : attn_mask,
            "label_ids"   : torch.tensor(label_id, dtype=torch.long),
        }


def build_dataloaders(args):
    tokenizer = AutoTokenizer.from_pretrained(args.text_model_name)

    train_ds = FinIntentDataset(train_df, tokenizer, args, "train")
    val_ds   = FinIntentDataset(val_df,   tokenizer, args, "val")
    test_ds  = FinIntentDataset(test_df,  tokenizer, args, "test")

    train_dl = DataLoader(train_ds, batch_size=args.train_batch_size,
                          shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=args.eval_batch_size,
                          shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=args.eval_batch_size,
                          shuffle=False, num_workers=2, pin_memory=True)

    return train_dl, val_dl, test_dl, tokenizer

print("✅ Dataset class defined.")


# ==============================================================
# CELL 9: ViSK Model
# ==============================================================
# @title ViSK Model (Video Branch)
# Directly adapted from the paper's model.py

class PerturbationNet(nn.Module):
    """
    Generates instance-aware σ for each spatiotemporal block.
    Input : (B*J, C, T, H, W)
    Output: (B,  J, 1, T, H, W) — one σ per block
    """
    def __init__(self, args, in_channels):
        super().__init__()
        self.conv1    = nn.Conv3d(in_channels, args.channels_1, 3, padding=1)
        self.conv2    = nn.Conv3d(args.channels_1, args.channels_2, 3, padding=1)
        self.conv3    = nn.Conv3d(args.channels_2, 1, 3, padding=1)
        self.relu     = nn.ReLU()
        self.softplus = nn.Softplus()
        self.dropout  = nn.Dropout3d(args.perturb_dropout)

    def forward(self, x):
        B, J, C, T, H, W = x.shape
        x = x.view(B * J, C, T, H, W)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.dropout(x)
        sigma = self.softplus(self.conv3(x))       # (B*J, 1, T, H, W)
        return sigma.view(B, J, 1, T, H, W)


class NoiseLayer(nn.Module):
    def __init__(self, args, dim_c):
        super().__init__()
        self.alpha           = args.sigma_init_decay
        self.perturbation_net= PerturbationNet(args, in_channels=dim_c)

    def forward(self, x_expanded, unit_noise):
        """
        x_expanded : (B, J, C, T, H, W) — repeated original features
        unit_noise  : (B, J, C, T, H, W) — sampled N(0,I)
        Returns perturbed features (B, J, C, T, H, W)
        """
        eps   = 1e-6
        sigma = self.perturbation_net(x_expanded)          # (B,J,1,T,H,W)
        perturb = sigma * unit_noise                        # (B,J,C,T,H,W)

        # ── Scaling (eq. 8–9 in paper) ───────────────────────
        p_norm  = perturb.norm(p=2, dim=2, keepdim=True)   # (B,J,1,T,H,W)
        i_norm  = x_expanded.norm(p=2, dim=2, keepdim=True)
        p_norm  = torch.where(p_norm == 0,
                              torch.ones_like(p_norm), p_norm)
        thresh  = (i_norm / (p_norm + eps)) * self.alpha
        thresh  = torch.clamp(thresh, max=1.0)

        output  = x_expanded + perturb * thresh
        return output


class ViSK(nn.Module):
    """
    Video-only branch of ViSK.
    forward(x, unit_noise) → (logits, features)
    """
    def __init__(self, args):
        super().__init__()
        self.iterations  = args.iterations
        self.num_features= args.embed_dim * 2 ** (len(args.depths) - 1)

        # ── Video Swin Transformer backbone ──────────────────
        block         = partial(SwinTransformerBlock,
                                attn_layer=ShiftedWindowAttention3d)
        norm_layer    = partial(nn.LayerNorm, eps=1e-5)

        self.patch_embed = PatchEmbed3d(
            patch_size=args.patch_size,
            embed_dim =args.embed_dim,
            norm_layer=norm_layer,
        )
        self.pos_drop = nn.Dropout(p=args.dropout)

        layers, total_blocks, block_id = [], sum(args.depths), 0
        for i_stage, depth in enumerate(args.depths):
            stage = []
            dim   = args.embed_dim * 2 ** i_stage
            for i_layer in range(depth):
                sd_prob = args.stochastic_depth_prob * float(block_id) / max(total_blocks - 1, 1)
                stage.append(block(
                    dim,
                    args.num_heads[i_stage],
                    window_size       = args.window_size,
                    shift_size        = [0 if i_layer % 2 == 0
                                         else w // 2
                                         for w in args.window_size],
                    mlp_ratio         = args.mlp_ratio,
                    dropout           = args.dropout,
                    attention_dropout = args.attention_dropout,
                    stochastic_depth_prob=sd_prob,
                    norm_layer        = norm_layer,
                    attn_layer        = ShiftedWindowAttention3d,
                ))
                block_id += 1
            layers.append(nn.Sequential(*stage))
            if i_stage < len(args.depths) - 1:
                layers.append(PatchMerging(dim, norm_layer))

        self.features = nn.Sequential(*layers)
        self.norm     = norm_layer(self.num_features)

        # ── Perturbation + Quantification ────────────────────
        self.noise_layer = NoiseLayer(args, dim_c=self.num_features)

        # Two heads: discovery MLP (fc) + classification head (cls_head)
        self.fc       = nn.Linear(self.num_features, args.num_labels)
        self.cls_head = nn.Linear(self.num_features, args.num_labels)
        self.avgpool  = nn.AdaptiveAvgPool1d(1)

    def forward(self, x, unit_noise):
        """
        x          : (B, T, C, H, W)
        unit_noise : (B, J, C_feat, t, h, w)
        """
        # ── Feature extraction ────────────────────────────────
        x = x.permute(0, 2, 1, 3, 4).float()  # (B, C, T, H, W)
        x = self.patch_embed(x)                 # (B, t, h, w, d)
        x = self.pos_drop(x)
        x = self.features(x)                    # (B, t, h, w, d)
        x = self.norm(x)
        x = x.permute(0, 4, 1, 2, 3)           # (B, d, t, h, w)

        # ── Flatten blocks ────────────────────────────────────
        B, C, T, H, W = x.shape
        f = torch.flatten(x, 2).permute(0, 2, 1)  # (B, t*h*w, C)
        y = self.fc(f)                              # (B, t*h*w, num_labels)

        # ── Adaptive perturbation ─────────────────────────────
        x_exp = x.unsqueeze(1).expand(B, self.iterations,
                                      C, T, H, W).contiguous()
        x_per = self.noise_layer(x_exp, unit_noise)  # (B,J,C,T,H,W)

        # ── Gradient-based quantification (eq. 10) ───────────
        f_per  = torch.flatten(x_per, 3).permute(0, 1, 3, 2)
        # (B, J, t*h*w, C)
        f_per  = f_per.view(B, self.iterations, -1, f.size(-1))

        delta_f = torch.abs(f_per - f.unsqueeze(1)).mean(dim=-1) + 1e-6
        y_per   = self.fc(f_per)
        delta_y = F.mse_loss(y_per, y.unsqueeze(1),
                             reduction="none").sum(dim=-1)

        kappa = (delta_y / delta_f)               # (B, J, t*h*w)
        kappa = kappa.mean(dim=1).unsqueeze(2).detach()  # (B, t*h*w, 1)

        # ── Weighted aggregation ──────────────────────────────
        f_weighted = (f * kappa).permute(0, 2, 1)  # (B, C, t*h*w)
        f_pooled   = self.avgpool(f_weighted).squeeze(-1)  # (B, C)

        logits   = self.cls_head(f_pooled)
        return logits, f_pooled


print("✅ ViSK model class defined.")


# ==============================================================
# CELL 10: Text Branch (MuRIL / multilingual BERT)
# ==============================================================
# @title Text Encoder for Hinglish

class TextEncoder(nn.Module):
    """
    Wraps MuRIL (or any BERT-style model) and returns the
    [CLS] representation projected to `out_dim`.
    """
    def __init__(self, model_name: str, out_dim: int, freeze: bool = False):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(model_name)
        hidden        = self.bert.config.hidden_size
        self.proj     = nn.Linear(hidden, out_dim)
        self.dropout  = nn.Dropout(0.1)
        if freeze:
            for p in self.bert.parameters():
                p.requires_grad_(False)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask)
        cls_rep = out.last_hidden_state[:, 0, :]     # (B, hidden)
        return self.dropout(self.proj(cls_rep))       # (B, out_dim)


print("✅ TextEncoder defined.")


# ==============================================================
# CELL 11: Multimodal Fusion Model
# ==============================================================
# @title FinIntentModel — fuses video (ViSK) + text (MuRIL)

class FinIntentModel(nn.Module):
    """
    Late-fusion: video features from ViSK + text features
    from MuRIL are concatenated and classified.
    """
    def __init__(self, args, video_feat_dim: int):
        super().__init__()
        self.visk         = ViSK(args)
        text_proj_dim     = 256
        self.text_encoder = TextEncoder(
            model_name=args.text_model_name,
            out_dim=text_proj_dim,
        )
        fusion_dim = video_feat_dim + text_proj_dim
        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Dropout(0.1),
            nn.Linear(fusion_dim, args.num_labels),
        )

    def forward(self, video_feats, unit_noise,
                input_ids, attn_mask):
        v_logits, v_feat = self.visk(video_feats, unit_noise)
        t_feat           = self.text_encoder(input_ids, attn_mask)
        fused            = torch.cat([v_feat, t_feat], dim=-1)
        logits           = self.classifier(fused)
        return logits, fused


print("✅ FinIntentModel (multimodal) defined.")


# ==============================================================
# CELL 12: Metrics Helper
# ==============================================================
# @title Metrics

def compute_metrics(y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average="macro",   zero_division=0)
    wf1  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    prec = precision_score(y_true, y_pred, average="macro",   zero_division=0)
    rec  = recall_score(y_true, y_pred,    average="macro",   zero_division=0)
    wp   = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy"   : round(acc  * 100, 2),
        "f1"         : round(f1   * 100, 2),
        "weighted_f1": round(wf1  * 100, 2),
        "precision"  : round(prec * 100, 2),
        "recall"     : round(rec  * 100, 2),
        "weighted_p" : round(wp   * 100, 2),
    }

print("✅ Metrics helper defined.")


# ==============================================================
# CELL 13: Early Stopping
# ==============================================================
# @title EarlyStopping

class EarlyStopping:
    def __init__(self, patience: int = 5, mode: str = "max"):
        self.patience    = patience
        self.mode        = mode
        self.best_score  = -np.inf if mode == "max" else np.inf
        self.counter     = 0
        self.early_stop  = False
        self.best_model  = None

    def __call__(self, score, model):
        improved = (score > self.best_score) if self.mode == "max" \
                   else (score < self.best_score)
        if improved:
            self.best_score = score
            self.counter    = 0
            import copy
            self.best_model = copy.deepcopy(model)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

print("✅ EarlyStopping defined.")


# ==============================================================
# CELL 14: Training & Evaluation Loop (REPLACEMENT)
# ==============================================================
# @title Train / Eval functions

def make_unit_noise(model, batch_size, args):
    """
    Fixed: The noise tensor must match the spatiotemporal dimensions
    of the feature map after the Swin backbone's downsampling.
    """
    # Channel dimension at the final stage
    dim_c = args.embed_dim * 2 ** (len(args.depths) - 1)

    # Temporal dimension: 16 input frames / 2 (initial patch depth) = 8
    # Video Swin Tiny doesn't typically reduce T further in later stages.
    t_feat = args.num_frames // args.patch_size[0]

    # Spatial dimension: 224 / 4 (initial patch) / 2 / 2 / 2 (three stages of merging) = 7
    hw_feat = args.frame_size // 32

    return torch.randn(batch_size, args.iterations,
                       dim_c, t_feat, hw_feat, hw_feat).to(DEVICE)

# Note: Ensure run_epoch and other functions in Cell 14 remain the same.
def run_epoch(model, loader, optimizer, scheduler,
              criterion, args, train: bool):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc="Train" if train else "Eval", leave=False):
            video_feats = batch["video_feats"].to(DEVICE)
            input_ids   = batch["input_ids"].to(DEVICE)
            attn_mask   = batch["attn_mask"].to(DEVICE)
            label_ids   = batch["label_ids"].to(DEVICE)

            B            = video_feats.size(0)
            unit_noise   = make_unit_noise(model, B, args)

            logits, _    = model(video_feats, unit_noise,
                                 input_ids, attn_mask)
            loss         = criterion(logits, label_ids)

            if train:
                optimizer.zero_grad()
                loss.backward()
                if args.grad_clip > 0:
                    nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                optimizer.step()
                scheduler.step()

            total_loss  += loss.item() * B
            preds        = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(label_ids.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    metrics  = compute_metrics(np.array(all_labels), np.array(all_preds))
    metrics["loss"] = round(avg_loss, 4)
    return metrics

print("✅ Training loop defined.")


# ==============================================================
# CELL 15: Build Model + Optimizer
# ==============================================================
# @title Build Model, Optimizer, Scheduler

# ── Compute video feature dim ─────────────────────────────────
video_feat_dim = args.embed_dim * 2 ** (len(args.depths) - 1)
args.feat_size = video_feat_dim

# ── Dataloaders ───────────────────────────────────────────────
print("Building dataloaders (first run downloads MuRIL ~500 MB)…")
train_dl, val_dl, test_dl, tokenizer = build_dataloaders(args)
print(f"✅ Dataloaders ready. Batches — train:{len(train_dl)} val:{len(val_dl)} test:{len(test_dl)}")

# ── Model ─────────────────────────────────────────────────────
model = FinIntentModel(args, video_feat_dim).to(DEVICE)

# Optionally load pretrained Swin weights (if you have them)
# checkpoint = torch.load(args.pretrained_video_path, map_location="cpu")
# model.visk.load_state_dict(checkpoint, strict=False)

param_groups = [
    {"params": [p for n, p in model.named_parameters()
                if "bert" not in n and p.requires_grad],
     "lr": args.lr},
    {"params": [p for n, p in model.named_parameters()
                if "bert" in n and p.requires_grad],
     "lr": args.lr * 0.1},   # lower LR for pretrained BERT
]
optimizer = AdamW(param_groups, weight_decay=args.weight_decay)

total_steps  = len(train_dl) * args.num_train_epochs
warmup_steps = int(total_steps * args.warmup_proportion)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model ready. Trainable params: {total_params/1e6:.1f}M")


# ==============================================================
# CELL 16: Training Loop
# ==============================================================
# @title Run Training

early_stop = EarlyStopping(patience=args.wait_patience, mode="max")
history    = {"train": [], "val": []}

print(f"\n🚀 Starting training for up to {args.num_train_epochs} epochs…\n")

for epoch in trange(args.num_train_epochs, desc="Epoch"):

    train_m = run_epoch(model, train_dl, optimizer, scheduler,
                        criterion, args, train=True)
    val_m   = run_epoch(model, val_dl,   optimizer, scheduler,
                        criterion, args, train=False)

    history["train"].append(train_m)
    history["val"].append(val_m)

    eval_score = val_m[args.eval_monitor]
    early_stop(eval_score, model)

    logger.info(
        f"Epoch {epoch+1:3d} | "
        f"train_loss={train_m['loss']:.4f} acc={train_m['accuracy']:.1f}% "
        f"wf1={train_m['weighted_f1']:.1f}% | "
        f"val_loss={val_m['loss']:.4f} acc={val_m['accuracy']:.1f}% "
        f"wf1={val_m['weighted_f1']:.1f}% | "
        f"best_wf1={early_stop.best_score:.1f}%"
    )

    if early_stop.early_stop:
        logger.info(f"⏹ Early stopping at epoch {epoch+1}")
        break

# Restore best model
model = early_stop.best_model
print(f"\n✅ Training complete. Best Val WF1 = {early_stop.best_score:.2f}%")


# ==============================================================
# CELL 17: Test Evaluation
# ==============================================================
# @title Evaluate on Test Set

test_m = run_epoch(model, test_dl, optimizer, scheduler,
                   criterion, args, train=False)

print("\n" + "="*55)
print("            TEST RESULTS")
print("="*55)
for k, v in test_m.items():
    if k != "loss":
        print(f"  {k:<15} : {v}")
print("="*55)


# ==============================================================
# CELL 18: Save Model & Results
# ==============================================================
# @title Save Checkpoint & History

import json

# Save model weights
ckpt_path = os.path.join(OUTPUT_DIR, "finintent_visk_best.pt")
torch.save(model.state_dict(), ckpt_path)
print(f"✅ Model saved → {ckpt_path}")

# Save training history
hist_path = os.path.join(OUTPUT_DIR, "training_history.json")
with open(hist_path, "w") as f:
    json.dump({"train": history["train"],
               "val"  : history["val"],
               "test" : test_m,
               "label_map": id2label}, f, indent=2)
print(f"✅ History saved → {hist_path}")


# ==============================================================
# CELL 19: Plot Training Curves
# ==============================================================
# @title Visualise Training (optional)

import matplotlib.pyplot as plt

epochs = range(1, len(history["train"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(epochs, [m["loss"]        for m in history["train"]], label="Train")
axes[0].plot(epochs, [m["loss"]        for m in history["val"]],   label="Val")
axes[0].set_title("Cross-Entropy Loss"); axes[0].legend()
axes[0].set_xlabel("Epoch")

# Weighted F1
axes[1].plot(epochs, [m["weighted_f1"] for m in history["train"]], label="Train")
axes[1].plot(epochs, [m["weighted_f1"] for m in history["val"]],   label="Val")
axes[1].set_title("Weighted F1 (%)"); axes[1].legend()
axes[1].set_xlabel("Epoch")

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"✅ Plot saved → {plot_path}")


# ==============================================================
# CELL 20: Per-Class Analysis (Confusion Matrix)
# ==============================================================
# @title Confusion Matrix & Per-Class Breakdown

from sklearn.metrics import confusion_matrix, classification_report

# Collect test predictions
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_dl, desc="Collecting predictions"):
        video_feats = batch["video_feats"].to(DEVICE)
        input_ids   = batch["input_ids"].to(DEVICE)
        attn_mask   = batch["attn_mask"].to(DEVICE)
        label_ids   = batch["label_ids"].to(DEVICE)
        B           = video_feats.size(0)
        unit_noise  = make_unit_noise(model, B, args)

        logits, _   = model(video_feats, unit_noise, input_ids, attn_mask)
        preds       = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(label_ids.cpu().numpy().tolist())

target_names = [id2label[i] for i in range(args.num_labels)]
print("\nClassification Report:")
print(classification_report(all_labels, all_preds,
                            target_names=target_names,
                            zero_division=0))

# ── Confusion matrix heatmap ──────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(args.num_labels)); ax.set_yticks(range(args.num_labels))
ax.set_xticklabels(target_names, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(target_names, fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("FinIntent Confusion Matrix (Test Set)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=150); plt.show()
print(f"✅ Confusion matrix saved → {cm_path}")


# ==============================================================
# CELL 21: Video-Only Baseline (Ablation)
# ==============================================================
# @title Video-Only Baseline (no text) — for ablation table

class VideoOnlyModel(nn.Module):
    def __init__(self, args, video_feat_dim):
        super().__init__()
        self.visk = ViSK(args)
        self.cls  = nn.Linear(video_feat_dim, args.num_labels)

    def forward(self, video_feats, unit_noise,
                input_ids=None, attn_mask=None):
        logits, feats = self.visk(video_feats, unit_noise)
        return logits, feats

# To run the video-only baseline:
# video_model = VideoOnlyModel(args, video_feat_dim).to(DEVICE)
# then call run_epoch with it
print("✅ VideoOnlyModel (ablation) defined. Uncomment above to run.")


# ==============================================================
# CELL 22: Inference on a Single Sample
# ==============================================================
# @title Single Sample Inference Demo

def predict_single(video_id: str, text: str, start_sec: float, end_sec: float):
    model.eval()
    video_path  = find_video_file(args.video_folder, video_id)
    video_feats = extract_frames(video_path, start_sec, end_sec,
                                 args.num_frames, args.frame_size)
    video_feats = video_feats.unsqueeze(0).to(DEVICE)

    enc       = tokenizer(text, max_length=args.max_seq_len,
                          padding="max_length", truncation=True,
                          return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attn_mask = enc["attention_mask"].to(DEVICE)

    unit_noise = make_unit_noise(model, 1, args)

    with torch.no_grad():
        logits, _ = model(video_feats, unit_noise, input_ids, attn_mask)
        probs     = F.softmax(logits, dim=-1).squeeze(0)
        pred_id   = probs.argmax().item()

    print(f"\nText  : {text}")
    print(f"Video : {video_id}")
    print(f"\n→ Predicted Intent : {id2label[pred_id]}")
    print(f"→ Confidence       : {probs[pred_id].item()*100:.1f}%")
    print("\nTop-3 predictions:")
    top3 = probs.topk(3)
    for prob, idx in zip(top3.values, top3.indices):
        print(f"   {id2label[idx.item()]:<20} {prob.item()*100:.1f}%")

# Example usage — replace with a real row from your dataset:
# predict_single("MM_EP1_S1_5",
#                "like mai job 10th me tha tab se job kar raha hu",
#                5.22, 5.45)
print("✅ Inference function ready. Call predict_single(...) to test.")

✅ All packages installed.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Paths configured.
   Excel : /content/drive/MyDrive/Capstone2/Dataset.xlsx
   Videos: /content/drive/MyDrive/Capstone2/Video
✅ Using device: cuda
✅ Args set.
Loaded 299 rows. Columns: ['Video ID', 'Hinglish Text', 'Hindi Text', 'Label', 'Start time', 'End time']
      Video ID                                      Hinglish Text  \
0  MM_EP1_S1_0  lekin wo apna face hume nahi dikhana chahte au...   
1  MM_EP1_S1_1  good afternoon, good afternoon, bahut achha la...   
2  MM_EP1_S1_2  sir mai basically UP se hu, mera birth delhi s...   

                                          Hindi Text      Label Start time  \
0  लेकिन वह अपना चेहरा हमें नहीं दिखाना चाहते और ...       Care   00:04:00   
1  गुड आफ्टरनून। गुड आफ्टरनून। बहुत अच्छा लग रहा ...      Thank   00:12:00   
2  सर मैं बेसिकली यूपी से हूं। मेरा बर्थ दिल्ली स...  Introduce   00:

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.13 GiB is allocated by PyTorch, and 297.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)